In [1]:
import os
import numpy as np
import cv2 as cv

In [2]:
sequences_root = './data/belgiumts/sequences/'
seq_id = 1
camera_id = 1
seq_root = os.path.join(sequences_root, f'Seq{seq_id:02d}')
video_files_path = os.path.join(seq_root, f'{camera_id:02d}')
gt_file_path = os.path.join(sequences_root, f'sequence{seq_id}_GT.txt')
camera_set_file_path = os.path.join(seq_root, 'camera_set.txt')
poses_file_path = os.path.join(seq_root, [x for x in os.listdir(seq_root) if x.endswith('.poses')][0])

In [3]:
with open(poses_file_path, 'r') as f:
    lines = f.readlines()


poses_lines = [l.strip() for l in lines[4:] if l.strip()]
poses = {int((parts := line.split())[0]): {
    'R_vehicle': np.array(parts[1:10], dtype=float).reshape(3, 3),
    't_vehicle': np.array(parts[10:13], dtype=float)
} for line in poses_lines}

In [4]:
R_cam_str = """
0.26074615509705534322 -0.23937308575402516109 0.93526037466509825968
-0.96532572405317029762 -0.077247832271888527966 0.24935721142673666906
0.012557431398436305625 -0.96784983247706957155 -0.25121507257882286224
"""
R_cam = np.array(R_cam_str.split(), dtype=float).reshape(3, 3)

K_str = """
1401.4490697891651507 -0.13022024734980200411 813.55302388514940048
0 1403.2649118299896145 598.60305944790320609
0 0 1
"""
K = np.array(K_str.split(), dtype=float).reshape(3, 3)

t_cam_str = """
1.506889599053904405 -0.3303040418872391637 -0.082612542681732570315
"""
t_cam = np.array(t_cam_str.split(), dtype=float)

distortion_coeffs_str = """
-0.19926509974069675502 0.13345850034658560124 0.061006889380811869794
"""
distorsion_coeffs_components = distortion_coeffs_str.split()
distortion_coeffs = np.array([distorsion_coeffs_components[0], distorsion_coeffs_components[1], 0, 0, distorsion_coeffs_components[2]], dtype=float)

In [5]:
poles = []

with open(gt_file_path, 'r') as f:
    lines = f.readlines()

i = 0
num_poles = int(lines[i].strip())
i += 1

for pole_idx in range(num_poles):
    # Skip empty lines
    while i < len(lines) and lines[i].strip() == '':
        i += 1

    pole_data = {}
    
    # Pole number and ID
    pole_info = lines[i].strip().split()
    pole_data['idx'] = int(pole_info[0])
    pole_data['id'] = int(pole_info[1])
    i += 1
    
    # 3D coordinates
    pole_coords = list(map(float, lines[i].strip().split()))
    pole_data['coords'] = pole_coords
    i += 1
    
    # Number of traffic signs and their types
    signs_data = lines[i].strip()[:-1].split(';')
    pole_data['num_signs'] = int(signs_data[0])
    pole_data['sign_types'] = signs_data[1:]
    pole_data['signs'] = [{'type': t, 'index': idx} for idx, t in enumerate(signs_data[1:], 1)]
    i += 1
    
    # Number of centers
    num_centers = int(lines[i].strip())
    i += 1
    
    # Skip center coordinates
    pole_centers = []
    for _ in range(num_centers):
        pole_centers.append(list(map(float, lines[i].strip().split())))
        i += 1
    pole_data['centers'] = pole_centers
    
    # Number of bounding boxes
    num_bboxes = int(lines[i].strip())
    i += 1
    
    bboxes = []
    # Parse each bounding box
    for _ in range(num_bboxes):
        # Bounding box coordinates
        bbox_coords = list(map(float, lines[i].strip().split()))
        # x1, y1, x2, y2 = bbox_coords
        y1, x1, y2, x2= bbox_coords
        i += 1
        
        # Camera, frame, order number
        metadata = lines[i].strip().split()
        camera = int(metadata[0])
        frame = int(metadata[1])
        sign_idx = int(metadata[2])
        i += 1
        
        # Class label
        class_label = lines[i].strip().rstrip(';')
        bboxes.append({
            'camera': camera,
            'frame': frame,
            'sign_idx': sign_idx,
            'bbox': [x1, y1, x2, y2],
            'class_label': class_label
        })
        i += 1

        sign = pole_data['signs'][sign_idx - 1]
        
        # 3D center coordinates
        sign['center_3d'] = list(map(float, lines[i].strip().split()))
        i += 1

    pole_data['bboxes'] = bboxes
    poles.append(pole_data)


signs = []
for pole in poles:
    for sign in pole['signs']:
        bboxes = [bbox for bbox in pole['bboxes'] if bbox['sign_idx'] == sign['index'] and bbox['camera'] == camera_id]
        signs.append({**sign, 'pole_idx': pole['idx'], 'center_3d': np.array(sign['center_3d']), 'bboxes': bboxes})

In [59]:
for sign in signs:
    if sign['pole_idx'] == 1:
        print(sign)

{'type': 'E1', 'index': 1, 'center_3d': array([1.47989236e+05, 2.02129906e+05, 8.41300000e+00]), 'pole_idx': 1, 'bboxes': []}
{'type': 'begin', 'index': 2, 'center_3d': array([1.47989236e+05, 2.02129906e+05, 7.85500000e+00]), 'pole_idx': 1, 'bboxes': []}


In [11]:
for imfile in sorted(os.listdir(video_files_path)):
    img = cv.imread(os.path.join(video_files_path, imfile))
    frame_num = int(imfile.split('.')[1])
    if frame_num not in poses:
        continue
    pose = poses[frame_num]
    for sign in signs:
        P_vehicle = pose['R_vehicle'].T @ (sign['center_3d'] - pose['t_vehicle'])
        P_cam = R_cam.T @ (P_vehicle - t_cam)
        if P_cam[2] <= 0:
            continue
        pts_2d, _ = cv.projectPoints(
            P_cam.reshape(1, 1, 3),
            np.zeros(3),
            np.zeros(3),
            K,
            distortion_coeffs
        )
        u, v = pts_2d.ravel()
        if 0 <= u < img.shape[1] and 0 <= v < img.shape[0]:
            img = cv.circle(img, (int(u), int(v)), 4, (0, 0, 255), -1)

        bboxes_in_frame = [bbox for bbox in sign['bboxes'] if bbox['frame'] == frame_num]

        if bboxes_in_frame:
            for bbox in bboxes_in_frame:
                x1, y1, x2, y2 = map(int, bbox['bbox'])
                center_x = (x1 + x2) // 2
                center_y = (y1 + y2) // 2
                img = cv.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                img = cv.circle(img, (center_x, center_y), 4, (0, 255, 0), -1)

    cv.imshow('img', img)
    key = cv.waitKey(33 if bboxes_in_frame else 100)
    if key == ord('q'):
        break

cv.destroyAllWindows()

In [7]:
# Validate projection against known GT bboxes for camera 1
# From GT: pole 2, camera 1, frame 32617, sign C3
# bbox (y1=69.97, x1=696.26, y2=126.55, x2=738.3) -> center u≈717, v≈98
# 3D center: (147973.144, 202115.16, 10.204)

test_cases = [
    {'frame': 32617, 'P_w': np.array([147973.144, 202115.16, 10.204]),
     'expected_u': 717, 'expected_v': 98, 'label': 'pole2 C3 f32617'},
    {'frame': 32629, 'P_w': np.array([147973.144, 202115.16, 10.163]),
     'expected_u': 1077, 'expected_v': 40, 'label': 'pole2 C3 f32629'},
]

print("=== Formula comparison ===")
for tc in test_cases:
    pose = poses[tc['frame']]
    P_vehicle = pose['R_vehicle'].T @ (tc['P_w'] - pose['t_vehicle'])

    # Current formula (WRONG): R_cam as vehicle->camera
    P_cam_A = R_cam @ P_vehicle + t_cam

    # Fixed formula: R_cam is camera->vehicle, so invert it
    P_cam_B = R_cam.T @ (P_vehicle - t_cam)

    for label, P_cam in [("R_cam @ Pv + t", P_cam_A), ("R_cam.T @ (Pv - t)", P_cam_B)]:
        if P_cam[2] > 0:
            u = (K[0, 0]*P_cam[0] + K[0, 1] *
                 P_cam[1] + K[0, 2]*P_cam[2]) / P_cam[2]
            v = (K[1, 1]*P_cam[1] + K[1, 2]*P_cam[2]) / P_cam[2]
            print(
                f"  {tc['label']} | {label:22s} | projected=({u:.1f}, {v:.1f}) | expected=({tc['expected_u']}, {tc['expected_v']})")
        else:
            print(f"  {tc['label']} | {label:22s} | BEHIND CAMERA")

=== Formula comparison ===
  pole2 C3 f32617 | R_cam @ Pv + t         | BEHIND CAMERA
  pole2 C3 f32617 | R_cam.T @ (Pv - t)     | projected=(701.3, 83.1) | expected=(717, 98)
  pole2 C3 f32629 | R_cam @ Pv + t         | BEHIND CAMERA
  pole2 C3 f32629 | R_cam.T @ (Pv - t)     | projected=(1066.2, 20.2) | expected=(1077, 40)


In [6]:
# --- Estimate 3D sign geometry from GT bboxes, then generate bboxes for all frames ---
# Fix 1: Optimize 3D center per-sign to minimize reprojection error across GT frames
# Fix 2: Use center+depth-scaled-size instead of corner projection (avoids perspective shift)

from scipy.optimize import least_squares

up_world = np.array([0.0, 0.0, 1.0])  # signs are vertical


def optimize_sign_center(sign):
    """Refine the 3D center to minimize reprojection error against all GT bbox centers."""
    gt_obs = []
    for bbox in sign['bboxes']:
        if bbox['frame'] not in poses:
            continue
        x1, y1, x2, y2 = bbox['bbox']
        gt_obs.append((bbox['frame'], (x1 + x2) / 2, (y1 + y2) / 2))

    if len(gt_obs) < 1:
        return sign['center_3d'].copy()

    c0 = sign['center_3d']

    # With 1 bbox (2 residuals), only optimize X,Y; with 2+ bboxes optimize all 3
    optimize_z = len(gt_obs) >= 2

    def residuals(params):
        if optimize_z:
            center_3d = params
        else:
            center_3d = np.array([params[0], params[1], c0[2]])
        res = []
        for frame, u_gt, v_gt in gt_obs:
            pose = poses[frame]
            P_v = pose['R_vehicle'].T @ (center_3d - pose['t_vehicle'])
            P_c = R_cam.T @ (P_v - t_cam)
            if P_c[2] <= 0:
                res.extend([1e3, 1e3])
                continue
            pts_2d, _ = cv.projectPoints(
                P_c.reshape(1, 1, 3), np.zeros(3), np.zeros(3), K, distortion_coeffs)
            u, v = pts_2d.ravel()
            res.extend([u - u_gt, v - v_gt])
        return res

    x0 = c0.copy() if optimize_z else c0[:2].copy()
    result = least_squares(residuals, x0, method='lm')
    if optimize_z:
        return result.x
    else:
        return np.array([result.x[0], result.x[1], c0[2]])


def project_sign_bbox(sign, frame_num, img_w=1628, img_h=1236):
    """Project 3D center, then apply depth-scaled physical size for the bbox."""
    geom = sign.get('geometry')
    if geom is None or frame_num not in poses:
        return None

    center = sign['center_3d_opt']
    pose = poses[frame_num]
    P_v = pose['R_vehicle'].T @ (center - pose['t_vehicle'])
    P_c = R_cam.T @ (P_v - t_cam)
    if P_c[2] <= 0:
        return None

    # Project center with distortion
    pts_2d, _ = cv.projectPoints(
        P_c.reshape(1, 1, 3), np.zeros(3), np.zeros(3), K, distortion_coeffs)
    u_c, v_c = pts_2d.ravel()

    # Depth-scaled half-sizes in pixels
    Z = P_c[2]
    hw_px = geom['half_w'] * K[0, 0] / Z
    hh_px = geom['half_h'] * K[1, 1] / Z

    # Bbox centered on projected center
    x1 = max(0, u_c - hw_px)
    y1 = max(0, v_c - hh_px)
    x2 = min(img_w, u_c + hw_px)
    y2 = min(img_h, v_c + hh_px)

    if x2 <= x1 or y2 <= y1:
        return None

    return [x1, y1, x2, y2]


# --- Run for all signs ---
for sign in signs:
    if not sign['bboxes']:
        sign['geometry'] = None
        sign['center_3d_opt'] = sign['center_3d'].copy()
        continue

    # Step 1: Optimize 3D center
    sign['center_3d_opt'] = optimize_sign_center(sign)
    shift = sign['center_3d_opt'] - sign['center_3d']
    shift_mm = np.linalg.norm(shift) * 1000

    # Step 2: Estimate physical half-size from GT bboxes using optimized center
    half_ws, half_hs = [], []
    for bbox in sign['bboxes']:
        if bbox['frame'] not in poses:
            continue
        pose = poses[bbox['frame']]
        P_v = pose['R_vehicle'].T @ (sign['center_3d_opt'] - pose['t_vehicle'])
        P_c = R_cam.T @ (P_v - t_cam)
        if P_c[2] <= 0:
            continue

        x1, y1, x2, y2 = bbox['bbox']
        Z = P_c[2]
        half_ws.append((x2 - x1) / 2 * Z / K[0, 0])
        half_hs.append((y2 - y1) / 2 * Z / K[1, 1])

    if not half_ws:
        sign['geometry'] = None
        continue

    sign['geometry'] = {
        'half_w': np.median(half_ws),
        'half_h': np.median(half_hs),
    }

    print(f"Sign {sign['type']:>10} (pole {sign['pole_idx']:>2}): "
          f"size ≈ {2*sign['geometry']['half_w']:.2f} x {2*sign['geometry']['half_h']:.2f} m, "
          f"center shift = {shift_mm:.0f} mm ({shift[0]:+.3f}, {shift[1]:+.3f}, {shift[2]:+.3f})")

print(f"\nReady to generate bboxes for {sum(1 for s in signs if s.get('geometry'))} / {len(signs)} signs")

Sign         C3 (pole  2): size ≈ 0.51 x 0.64 m, center shift = 201 mm (+0.141, +0.144, +0.000)
Sign        C29 (pole  2): size ≈ 0.53 x 0.69 m, center shift = 168 mm (+0.125, +0.111, +0.022)
Sign        B19 (pole  2): size ≈ 0.50 x 0.69 m, center shift = 135 mm (+0.091, +0.096, +0.027)
Sign        C21 (pole  2): size ≈ 0.52 x 0.70 m, center shift = 144 mm (+0.088, +0.109, +0.030)
Sign        C29 (pole  3): size ≈ 0.70 x 0.69 m, center shift = 1548 mm (-1.483, -0.400, +0.198)
Sign        X31 (pole  5): size ≈ 0.62 x 0.91 m, center shift = 369 mm (+0.341, +0.141, +0.011)
Sign        E9a (pole  5): size ≈ 0.41 x 0.60 m, center shift = 265 mm (+0.231, +0.128, +0.022)
Sign      einde (pole  5): size ≈ 0.12 x 0.39 m, center shift = 347 mm (+0.307, +0.154, +0.055)
Sign        X32 (pole  6): size ≈ 0.31 x 0.59 m, center shift = 63 mm (+0.062, +0.013, +0.008)
Sign        E9a (pole  8): size ≈ 0.43 x 0.60 m, center shift = 185 mm (+0.053, +0.177, +0.010)
Sign      begin (pole  8): size ≈ 0.12 x

In [7]:
# Visualize: generated bboxes (blue) vs GT bboxes (green)
for imfile in sorted(os.listdir(video_files_path)):
    img = cv.imread(os.path.join(video_files_path, imfile))
    frame_num = int(imfile.split('.')[1])
    if frame_num not in poses:
        continue

    for sign in signs:
        # Generated bbox (blue)
        gen_bbox = project_sign_bbox(sign, frame_num, img.shape[1], img.shape[0])
        if gen_bbox is not None:
            gx1, gy1, gx2, gy2 = map(int, gen_bbox)
            # img = cv.putText(img, f"{sign['type']} (pole {sign['pole_idx']})", (gx1, gy1), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            img = cv.rectangle(img, (gx1, gy1), (gx2, gy2), (255, 0, 0), 2)

        # GT bbox (green) for comparison
        for bbox in sign['bboxes']:
            if bbox['frame'] == frame_num:
                x1, y1, x2, y2 = map(int, bbox['bbox'])
                img = cv.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    cv.imshow('Blue=generated, Green=GT', img)
    if cv.waitKey(33) == ord('q'):
        break

cv.destroyAllWindows()

error: OpenCV(4.13.0) /io/opencv/modules/highgui/src/window.cpp:1301: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvShowImage'


In [30]:
# === DIAGNOSTIC: Pole 2 — compare BEFORE (raw 3D center) vs AFTER (optimized center) ===

pole2_signs = [s for s in signs if s['pole_idx'] == 2]
print(f"Pole 2: {len(pole2_signs)} signs — {[s['type'] for s in pole2_signs]}\n")

print(f"{'sign':>6} {'frame':>6} | {'gt_center':>20} | {'raw_proj (err)':>28} | {'opt_proj (err)':>28} | {'gen_bbox_center (err)':>28}")
print("-" * 140)

for sign in pole2_signs:
    for bbox in sign['bboxes']:
        frame = bbox['frame']
        if frame not in poses:
            continue
        pose = poses[frame]

        # GT bbox center
        x1, y1, x2, y2 = bbox['bbox']
        u_gt, v_gt = (x1 + x2) / 2, (y1 + y2) / 2

        # Raw 3D center projection
        P_v = pose['R_vehicle'].T @ (sign['center_3d'] - pose['t_vehicle'])
        P_c = R_cam.T @ (P_v - t_cam)
        pts_2d, _ = cv.projectPoints(P_c.reshape(1, 1, 3), np.zeros(3), np.zeros(3), K, distortion_coeffs)
        u_raw, v_raw = pts_2d.ravel()

        # Optimized center projection
        P_v2 = pose['R_vehicle'].T @ (sign['center_3d_opt'] - pose['t_vehicle'])
        P_c2 = R_cam.T @ (P_v2 - t_cam)
        pts_2d2, _ = cv.projectPoints(P_c2.reshape(1, 1, 3), np.zeros(3), np.zeros(3), K, distortion_coeffs)
        u_opt, v_opt = pts_2d2.ravel()

        # Generated bbox center
        gen = project_sign_bbox(sign, frame)
        if gen:
            u_gen, v_gen = (gen[0] + gen[2]) / 2, (gen[1] + gen[3]) / 2
            gen_str = f"({u_gen:7.1f},{v_gen:6.1f}) ({u_gen-u_gt:+5.1f},{v_gen-v_gt:+5.1f})"
        else:
            gen_str = "N/A"

        raw_err = f"({u_raw:7.1f},{v_raw:6.1f}) ({u_raw-u_gt:+5.1f},{v_raw-v_gt:+5.1f})"
        opt_err = f"({u_opt:7.1f},{v_opt:6.1f}) ({u_opt-u_gt:+5.1f},{v_opt-v_gt:+5.1f})"
        print(f"{sign['type']:>6} {frame:>6} | ({u_gt:7.1f},{v_gt:6.1f}) | {raw_err} | {opt_err} | {gen_str}")
    print()

Pole 2: 4 signs — ['C3', 'C29', 'B19', 'C21']

  sign  frame |            gt_center |               raw_proj (err) |               opt_proj (err) |        gen_bbox_center (err)
--------------------------------------------------------------------------------------------------------------------------------------------
    C3  32060 | ( 1304.7,  34.8) | ( 1307.6,  39.0) ( +2.9, +4.1) | ( 1305.0,  36.5) ( +0.3, +1.7) | ( 1305.0,  36.5) ( +0.3, +1.7)
    C3  32617 | (  717.3,  98.3) | (  704.2,  99.5) (-13.1, +1.3) | (  717.4,  98.6) ( +0.1, +0.4) | (  717.4,  98.6) ( +0.1, +0.4)
    C3  32629 | ( 1077.4,  40.3) | ( 1057.4,  40.1) (-20.0, -0.3) | ( 1077.5,  38.6) ( +0.1, -1.8) | ( 1077.5,  38.9) ( +0.1, -1.5)

   C29  32060 | ( 1300.5, 198.1) | ( 1303.9, 203.7) ( +3.4, +5.7) | ( 1300.6, 201.3) ( +0.1, +3.2) | ( 1300.6, 201.3) ( +0.1, +3.2)
   C29  32629 | ( 1064.9, 226.1) | ( 1049.8, 226.7) (-15.1, +0.7) | ( 1065.3, 224.5) ( +0.4, -1.6) | ( 1065.3, 224.5) ( +0.4, -1.6)
   C29  32617 | (  71